# 대형언어모델 실습

**Large Language Model · LLM**

대규모 텍스트 등으로 학습해 언어를 처리하고 생성하는 모델. 출력의 사실성은 별도 확인이 필요하다.

소재 분야에서 이해하기: 논문에서 합성 조건을 추출한 뒤 원문과 대조한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [Google ML 용어집](https://developers.google.com/machine-learning/glossary)

## 1. API 없이 언어모델의 기본 감각 익히기

외부 호출 없이, 문장을 토큰으로 나누고 다음 단어 확률을 세어보는 아주 작은 모델을 만듭니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

text = ("소성 온도를 올리면 결정립이 커진다 . 결정립이 커지면 강도가 낮아진다 . "
        "소성 온도를 낮추면 결정립이 작아진다 . 결정립이 작아지면 강도가 높아진다 . ")
tokens = text.split()
print('토큰 %d개, 서로 다른 토큰 %d개' % (len(tokens), len(set(tokens))))
print(tokens[:12])

In [ ]:
from collections import Counter, defaultdict

next_words = defaultdict(Counter)
for a, b in zip(tokens, tokens[1:]):
    next_words[a][b] += 1

for word in ('결정립이', '강도가'):
    counts = next_words[word]
    total = sum(counts.values())
    print(word, '->', {k: round(v / total, 2) for k, v in counts.items()})

## 2. 이어붙여 생성해보기

In [ ]:
def generate(start, steps=8, seed=1):
    local = np.random.default_rng(seed)
    out = [start]
    for _ in range(steps):
        counts = next_words[out[-1]]
        if not counts:
            break
        words = list(counts); weights = np.array([counts[w] for w in words], float)
        out.append(local.choice(words, p=weights / weights.sum()))
    return ' '.join(out)

print(generate('소성'))
print(generate('결정립이', seed=3))

## 3. 해석

실제 LLM은 훨씬 큰 데이터와 문맥을 쓰지만, 다음 토큰의 확률을 예측한다는 골격은 같습니다.
확률로 이어붙이는 구조이므로 **그럴듯하지만 사실이 아닌 문장**이 나올 수 있습니다.
논문에서 값을 뽑는 데 쓸 때는 항상 원문과 대조해야 합니다.

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#llm)을 여세요.